# Marcos do agente de reunião pt-BR — baixar, subir e continuar

Gestão de **checkpoints (marcos)** do modelo treinado no Tinker:

1. **Inspecionar** o estado local (`.art/`) e listar os checkpoints vivos na nuvem do Tinker;
2. **Baixar** o adaptador LoRA de um marco (formato peft) para disco;
3. **Subir** o marco para um repositório privado no Hugging Face (backup durável — checkpoints no Tinker podem expirar);
4. **Restaurar** um backup e **continuar o treino de onde parou**.

O que cada artefato permite: o **checkpoint na nuvem do Tinker** (com otimizador) é o único que retoma o treino RL exatamente de onde parou; o **adaptador LoRA baixado** serve para merge/serving/SFT em qualquer lugar; o **`state.json`** é o elo local que liga o `name` do modelo aos runs do Tinker.

Chaves: `TINKER_API_KEY` (obrigatória) e `HF_TOKEN` (só para as seções de backup/restauração no HF).

In [ ]:
%pip uninstall -q -y openpipe-art
%pip install -q "openpipe-art[tinker] @ git+https://github.com/OpenPipe/ART.git@3492eb9a9ff1" python-dotenv huggingface_hub

In [ ]:
import os

# Somente carrega (.env na raiz do repo -> secrets do Colab); sem prompts.
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except ImportError:
    print("python-dotenv ausente — rode a célula de instalação e reinicie o kernel.")
try:
    from google.colab import userdata
    for k in ("TINKER_API_KEY", "HF_TOKEN"):
        if not os.environ.get(k):
            try:
                os.environ[k] = userdata.get(k) or ""
            except Exception:
                pass
except ImportError:
    pass

assert os.environ.get("TINKER_API_KEY"), "TINKER_API_KEY é obrigatória"
if not os.environ.get("HF_TOKEN"):
    print("HF_TOKEN ausente — as seções de backup/restauração no HF ficarão indisponíveis.")

## 1. Estado local do modelo

In [ ]:
import json

PROJECT = "tag-ai-meeting-agent"
MODEL_NAME = "meeting-agent-ptbr-002"   # o modelo cujos marcos você quer gerir
BASE_MODEL = "Qwen/Qwen3.6-35B-A3B"

from art.utils.output_dirs import get_default_art_path

MODEL_DIR = f"{get_default_art_path()}/{PROJECT}/models/{MODEL_NAME}"
STATE_PATH = f"{MODEL_DIR}/state.json"

state = {}
if os.path.exists(STATE_PATH):
    state = json.load(open(STATE_PATH))
    print(f"Último step treinado: {state.get('latest_step')}")
    print("Runs do Tinker (linhagem):")
    for r in state.get("tinker_run_ids", []):
        print("  -", r)
else:
    print(f"Sem estado local em {STATE_PATH} — use a seção 5 para restaurar de um backup.")

## 2. Listar marcos vivos na nuvem do Tinker

In [ ]:
import tinker

service_client = tinker.ServiceClient()          # usa TINKER_API_KEY
rest_client = service_client.create_rest_client()

checkpoints = []   # (run_id, checkpoint_id, tipo, criado_em, tinker_path)
for run_id in state.get("tinker_run_ids", []):
    try:
        resp = await rest_client.list_checkpoints_async(run_id)
    except Exception as e:
        print(f"{run_id}: {type(e).__name__} — run possivelmente expirado")
        continue
    for c in resp.checkpoints:
        kind = str(getattr(c, "checkpoint_type", "?"))
        prefix = "sampler_weights" if "sampler" in kind else "weights"
        path = getattr(c, "path", None) or f"tinker://{run_id}/{prefix}/{c.checkpoint_id}"
        checkpoints.append((run_id, c.checkpoint_id, kind, str(getattr(c, "time", "")), path))

print(f"{len(checkpoints)} checkpoints encontrados:")
for run_id, cid, kind, ts, path in checkpoints:
    print(f"  [{kind:>26}] {cid:<14} {ts[:19]}  {path}")

## 3. Baixar um marco (adaptador LoRA)

Escolha o step em `STEP` (`None` = mais recente). O servidor do Tinker monta o arquivo sob demanda — a primeira tentativa pode estourar timeout; o loop re-tenta.

In [ ]:
from tinker_cookbook import weights
import time

STEP = None   # ex.: 10 para o marco step_000010; None = último treinado

step = STEP if STEP is not None else int(state.get("latest_step", 0))
CKPT = f"step_{step:06d}"
# prioriza o path listado na seção 2; senão constrói pelo último run
listed = [p for _, cid, kind, _, p in checkpoints if cid == CKPT and "sampler" in kind]
sampler_path = listed[-1] if listed else \
    f"tinker://{state['tinker_run_ids'][-1]}/sampler_weights/{CKPT}"

DEST = f"milestones/{MODEL_NAME}/{CKPT}"
adapter_dir = None
for attempt in range(6):
    try:
        adapter_dir = weights.download(tinker_path=sampler_path,
                                       output_dir=f"{DEST}/adapter")
        break
    except Exception as e:
        print(f"Tentativa {attempt + 1}/6: {type(e).__name__} — aguardando 60s…")
        time.sleep(60)
assert adapter_dir, "Download não ficou pronto — re-execute em alguns minutos."

manifest = {
    "model_name": MODEL_NAME, "project": PROJECT, "base_model": BASE_MODEL,
    "step": step, "checkpoint": CKPT, "tinker_path": sampler_path,
    "tinker_run_ids": state.get("tinker_run_ids", []),
}
os.makedirs(DEST, exist_ok=True)
json.dump(manifest, open(f"{DEST}/manifest.json", "w"), indent=2, ensure_ascii=False)
print(f"Marco salvo em {DEST}/ (adaptador + manifest)")

## 4. Subir o marco para o Hugging Face (backup durável)

Repositório **privado** com um diretório por marco + o `state.json` (o elo para continuar o treino em outra máquina). Edite `HF_USER`.

In [ ]:
from huggingface_hub import HfApi

HF_USER = "seu-usuario"   # <-- EDITE
REPO = f"{HF_USER}/meeting-agent-ptbr-milestones"

api = HfApi(token=os.environ["HF_TOKEN"])
api.create_repo(REPO, private=True, exist_ok=True)
api.upload_folder(repo_id=REPO, folder_path=DEST,
                  path_in_repo=f"{MODEL_NAME}/{CKPT}")
if os.path.exists(STATE_PATH):
    api.upload_file(repo_id=REPO, path_or_fileobj=STATE_PATH,
                    path_in_repo=f"{MODEL_NAME}/art_state/state.json")
print(f"Backup publicado: https://huggingface.co/{REPO} ({MODEL_NAME}/{CKPT})")

## 5. Restaurar um backup (máquina nova ou `.art/` perdido)

Baixa os marcos do HF e recoloca o `state.json` em `.art/` — a partir daí o notebook de treino reconecta pelo `name` normalmente. **Não sobrescreve** um `state.json` existente, a menos que `RESTORE_FORCE = True`.

In [ ]:
from huggingface_hub import snapshot_download
import shutil

RESTORE_FORCE = False

local = snapshot_download(repo_id=REPO, repo_type="model",
                          allow_patterns=[f"{MODEL_NAME}/*"],
                          token=os.environ["HF_TOKEN"])
backup_state = f"{local}/{MODEL_NAME}/art_state/state.json"
print("Backup baixado em:", local)

if os.path.exists(backup_state):
    if os.path.exists(STATE_PATH) and not RESTORE_FORCE:
        print("state.json local já existe — nada sobrescrito "
              "(defina RESTORE_FORCE = True para forçar).")
    else:
        os.makedirs(MODEL_DIR, exist_ok=True)
        shutil.copy(backup_state, STATE_PATH)
        print(f"state.json restaurado em {STATE_PATH}")
else:
    print("Backup não contém art_state/state.json.")

## 6. Continuar de onde parou

Com o `state.json` no lugar, a retomada é o fluxo normal do **notebook de treino**: secrets → seções 1–5 → **seção 4** (o `register` reconecta pelo `name`) → **item 8** (o loop começa em `model.get_step()`). A célula abaixo só faz a verificação.

⚠️ Retomar o treino RL exige o **checkpoint de treino vivo na nuvem do Tinker** (com otimizador). Se os runs expiraram, o adaptador baixado ainda serve para merge/serving/SFT, mas o RL recomeça de um run novo.

In [ ]:
import art
from art.tinker_native import TinkerNativeBackend

backend = TinkerNativeBackend()
model = art.TrainableModel(name=MODEL_NAME, project=PROJECT, base_model=BASE_MODEL)
await model.register(backend)

step = await model.get_step()
print(f"Pronto para continuar: próximo step = {step}")
print("Abra o notebook de treino, ajuste NUM_STEPS e rode o item 8.")